In [2]:
# Importing the global libraries
import importlib, sys, os, pandas as pd
# from dotenv import load_dotenv
import pyspark.sql.types as T
import pyspark.sql as sql
from pyspark.sql import SparkSession, functions as F
import numpy as np
import datetime as dt

spark = (
	SparkSession.builder
		.appName("hashmap_stations")
		.getOrCreate()
)

#Mandatory
importlib.reload(importlib)
%load_ext autoreload
%autoreload 2

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/24 13:19:32 WARN Utils: Your hostname, BEBEL-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/24 13:19:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 13:19:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/24 13:19:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
stations_df = spark.read.csv("sumo_data/stations.csv", header=True, sep=";", inferSchema=True)
station_to_station_df = spark.read.csv("sumo_data/station_to_station.csv", header=True, sep=";", inferSchema=True)
punctuality_data_df = spark.read.csv("sumo_data/punctuality/202501.csv", header=True, sep=";", inferSchema=True)

In [4]:
stations_df.printSchema()
stations_df.show(5)

root
 |-- ID: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Symbolic_name: string (nullable = true)
 |-- Name_FR_short: string (nullable = true)
 |-- Name_FR_full: string (nullable = true)

+----+---------+--------+-------------+-------------+------------+
|  ID|    Geo_x|   Geo_y|Symbolic_name|Name_FR_short|Name_FR_full|
+----+---------+--------+-------------+-------------+------------+
| 841|50.932897|4.216581|          LHL|       Mollem|      Mollem|
| 246|50.527312|3.526278|          FCA|   Callenelle|  Callenelle|
| 457|50.726468|4.513842|          MGV|       Genval|      Genval|
| 782|50.614049|3.800495|          FFA|       Maffle|      Maffle|
|1102|50.528178|5.219617|          LHY|       Statte|      Statte|
+----+---------+--------+-------------+-------------+------------+
only showing top 5 rows


In [5]:
station_to_station_df.printSchema()
station_to_station_df.show(5)

root
 |-- Departure_station_id: integer (nullable = true)
 |-- Arrival_station_id: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- Coordinates: string (nullable = true)

+--------------------+------------------+---------+--------+--------+--------------------+
|Departure_station_id|Arrival_station_id|    Geo_x|   Geo_y|Distance|         Coordinates|
+--------------------+------------------+---------+--------+--------+--------------------+
|                 841|               821|50.943277|4.221739|    2.46|[[50.932897, 4.21...|
|                 246|               958|50.520529|3.559228|    4.96|[[50.513613, 3.59...|
|                 457|               672|50.732192|4.505572|    1.74|[[50.73811, 4.497...|
|                 782|                77|50.621327|3.788608|    2.34|[[50.614049, 3.80...|
|                1102|               592|50.526327|5.226778|    1.18|[[50.527376, 5.23...|
+----

In [6]:
punctuality_data_df.printSchema()
punctuality_data_df.show(5)

root
 |-- TRAIN_NO: integer (nullable = true)
 |-- RELATION: string (nullable = true)
 |-- TRAIN_SERV: string (nullable = true)
 |-- STOPPING_PLACE_ID: integer (nullable = true)
 |-- THOP1_COD: string (nullable = true)
 |-- LINE_NO_DEP: string (nullable = true)
 |-- DELAY_ARR: integer (nullable = true)
 |-- DELAY_DEP: integer (nullable = true)
 |-- CIRC_TYP: integer (nullable = true)
 |-- RELATION_DIRECTION: string (nullable = true)
 |-- LINE_NO_ARR: string (nullable = true)
 |-- REAL_DATE_ARR: string (nullable = true)
 |-- REAL_DATE_DEP: string (nullable = true)
 |-- PLANNED_DATETIME_ARR: timestamp (nullable = true)
 |-- PLANNED_DATETIME_DEP: timestamp (nullable = true)

+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECT

In [7]:
window = sql.Window.partitionBy("TRAIN_NO","REAL_DATE_DEP") \
	.orderBy("PLANNED_DATETIME_DEP")

punctuality_data_df = (
	punctuality_data_df
	.withColumn(
		"NEXT_STOPPING_PLACE_ID",
		F.lead("STOPPING_PLACE_ID").over(window)
	)
	.withColumn("PREV_STOPPING_PLACE_ID", F.lag("STOPPING_PLACE_ID").over(window))
)

punctuality_data_df = (
	punctuality_data_df.alias("a")
	# .join(
	# 	stations_df.select("ID").alias("b"), 
	# 	F.col("a.NEXT_STOPPING_PLACE_ID") == F.col("b.ID"), 
	# 	"left")
	# .drop("ID")
	.sort("TRAIN_NO", "PLANNED_DATETIME_DEP", ascending=True)
)
punctuality_data_df.show(20)

+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECTION|LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|NEXT_STOPPING_PLACE_ID|PREV_STOPPING_PLACE_ID|
+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+----------------------+
|      10|     ICE| SNCB/NMBS|              220|     NULL|       NULL|      -58|     NULL|       1|ICE: FRANKFURT(MA...|        0/2|   01-01-2025|         NULL| 2025-01-01 21:35:00|                NULL|                   220|                  NULL|
|   

In [8]:
neighbors = punctuality_data_df.select("STOPPING_PLACE_ID", "NEXT_STOPPING_PLACE_ID", "PREV_STOPPING_PLACE_ID").distinct()
neighbors.show(20)

+-----------------+----------------------+----------------------+
|STOPPING_PLACE_ID|NEXT_STOPPING_PLACE_ID|PREV_STOPPING_PLACE_ID|
+-----------------+----------------------+----------------------+
|              744|                   745|                  1139|
|              155|                    31|                  1202|
|              866|                   590|                   863|
|              726|                    27|                   638|
|              220|                  2089|                   217|
|              139|                    37|                    37|
|                6|                   701|                   365|
|             1134|                  1270|                   686|
|              810|                  1224|                   811|
|             1048|                   221|                  1057|
|             1266|                    74|                   621|
|              684|                   210|                   684|
|         

In [9]:
stations_dict = {}
for row in station_to_station_df.collect():
	departure_station, arrival_station = row['Departure_station_id'], row['Arrival_station_id']
	if departure_station not in stations_dict:
		stations_dict[departure_station] = set()
	stations_dict[departure_station].add(arrival_station)
for key in stations_dict:
	print(f"{key} -> {stations_dict[key]}")

841 -> {74, 821}
246 -> {958, 807}
457 -> {672, 997}
782 -> {832, 77}
1102 -> {592, 118}
157 -> {184, 324, 1151}
1912 -> {1761, 380, 189}
143 -> {708, 1348}
901 -> {956, 421}
1761 -> {1912, 826, 1219, 380}
952 -> {989, 583}
578 -> {742, 1079}
754 -> {148, 438}
733 -> {835, 470}
619 -> {401, 971}
1146 -> {1176, 384}
493 -> {1230, 190}
220 -> {455, 415, 2089, 815, 725, 504, 217, 1021, 414, 223}
900 -> {384, 684}
494 -> {345, 1228}
427 -> {560, 1154}
107 -> {840, 708}
249 -> {480, 951}
560 -> {427, 868}
132 -> {9, 187, 686}
1185 -> {477, 1157}
347 -> {67, 910}
1278 -> {64, 819, 58, 1084, 30}
736 -> {404, 1149}
1060 -> {938, 1253}
1266 -> {74, 621}
1091 -> {1843, 1206}
1157 -> {992, 1185}
936 -> {739, 975, 855, 762, 252}
118 -> {25, 1102}
19 -> {1090, 523}
130 -> {449, 748}
27 -> {2011, 266, 208, 1202, 726, 1147}
818 -> {114, 788}
905 -> {1066, 188}
1176 -> {1146, 715}
764 -> {58, 37, 38, 1839}
8 -> {136, 797}
142 -> {732, 814}
728 -> {730, 726}
267 -> {266, 1159}
148 -> {754, 1031}
192 ->

In [ ]:
missing_stations = set()
missing_stations_dict = {}

for prev_s, s, next_s in neighbors.collect():
	if prev_s != s != next_s and prev_s is not None and s is not None and next_s is not None :
		if prev_s not in stations_dict or s not in stations_dict or next_s not in stations_dict :
			if prev_s not in stations_dict :
				missing_stations.add(prev_s)
			if s not in stations_dict :
				missing_stations.add(s)
			if next_s not in stations_dict :
				missing_stations.add(next_s)
			if prev_s not in missing_stations_dict :
				missing_stations_dict[prev_s] = set()
			missing_stations_dict[prev_s].add(s)
			if s not in missing_stations_dict :
				missing_stations_dict[s] = set()
			missing_stations_dict[s].add(next_s)		
for key in missing_stations_dict:
	print(f"{key} -> {missing_stations_dict[key]}")

744 -> {745, 1139, 412}
745 -> {744, 1139, 412, 664}
726 -> {421, 358, 2011, 1063, 1159, 638, 1202, 562, 1246, 731, 728, 221, 27, 1148, 733, 734}
27 -> {929, 266, 1202, 731, 638}
1048 -> {800, 1057, 674, 363, 2061, 533, 1979, 221}
221 -> {1057, 734}
748 -> {1073, 130, 1265, 749}
749 -> {192, 130, 138, 748, 1265, 1073}
523 -> {324, 1061, 233, 524, 117, 636}
117 -> {324, 37, 326, 1061, 523, 139, 715, 462, 524, 19, 984, 636}
1724 -> {320, 449, 1088, 37, 455, 335, 447}
455 -> {320, 335, 1211, 1724, 93, 447}
743 -> {1723, 212, 1262, 1263}
212 -> {1263}
506 -> {360, 232, 591}
591 -> {504, 128}
784 -> {673, 674, 471}
674 -> {673, 391, 201, 1744, 784, 790, 471}
673 -> {674, 391, 201, 784, 1744, 1048}
913 -> {610, 611, 995, 269, 1009, 767, 789, 313, 895}
789 -> {913, 269, 894, 895}
530 -> {361, 557, 22}
557 -> {361, 402, 530, 22}
391 -> {673, 674, 790}
139 -> {162, 42, 524, 117, 1724, 734}
162 -> {1088, 139, 748, 37}
733 -> {562, 835, 726, 734}
835 -> {734}
924 -> {554, 436, 2222}
2222 -> {1232

In [11]:
from collections import deque

def closest_anchors(G1: dict, G2: dict, new_nodes: list):
    V1 = set(G1.keys())
    V2 = set(G2.keys())
    anchors = V1 & V2  # ancres = noeuds présents dans les deux graphes

    if not anchors:
        return {n: [] for n in new_nodes}

    # dist[v] = distance minimale à une ancre
    dist = {}
    # best[v] = ensemble des ancres atteignant dist[v]
    best = {}

    q = deque()

    # init: toutes les ancres à distance 0
    for a in anchors:
        dist[a] = 0
        best[a] = {a}
        q.append(a)

    # BFS multi-source sur G2
    while q:
        u = q.popleft()
        for v in G2.get(u, []):
            cand = dist[u] + 1

            # trouvé une meilleure distance
            if v not in dist or cand < dist[v]:
                dist[v] = cand
                best[v] = set(best[u])
                q.append(v)

            # trouvé une distance égale -> on ajoute les ancres
            elif cand == dist[v]:
                best[v].update(best[u])

    # on retourne UNIQUEMENT les ancres à distance minimale (et rien au-delà)
    res = {}
    for n in new_nodes:
        res[n] = sorted(best[n]) if n in best else []
    return res

res = closest_anchors(stations_dict, missing_stations_dict, missing_stations)
for station in res :
	print(station, res[station])

1155 [404, 736, 1149, 1189]
516 [190, 493, 507, 1230]
264 [16, 700]
650 [868]
1419 [376, 539, 632, 637, 1084]
524 [139, 523, 1061]
269 [789, 894, 1062]
2061 [1048]
528 [70, 489, 723, 752, 968]
913 [313, 610, 611, 767, 789, 895, 995, 1009]
533 [77, 427, 719, 1048, 1154, 1260]
2199 [798, 961, 1254]
921 [535, 908, 920, 1136]
794 [413, 747, 791]
283 [604, 606, 649, 1005]
925 [31, 267, 421, 901, 1159]
1055 [191, 218, 227, 383, 812, 826]
800 [363, 1048, 1192, 1224]
1057 [221, 363, 1048, 1192, 1979]
674 [201, 391, 471, 673, 784, 790, 1048, 1744]
162 [139, 151, 1088]
33 [368, 648, 916, 1174, 1260]
164 [347, 458, 600, 910, 923]
675 [217, 376, 764, 1839]
551 [205, 342, 550, 1092, 1160]
1577 [424, 515, 620, 809, 870]
42 [139, 546, 554, 977]
937 [153, 252, 739, 855, 936]
428 [1067, 2011]
557 [22, 361, 402, 530]
2222 [436, 554, 924, 1232]
308 [82, 515, 620, 1125]
949 [210, 931, 1261]
950 [210, 1195]
698 [560, 649, 868]
1211 [455, 682]
1724 [139, 320, 335, 447, 449, 455]
1982 [75, 272, 287, 896, 104

In [12]:
for station in res : 
	check = [s in stations_dict for s in res[station]]
	print(station, check)


1155 [True, True, True, True]
516 [True, True, True, True]
264 [True, True]
650 [True]
1419 [True, True, True, True, True]
524 [True, True, True]
269 [True, True, True]
2061 [True]
528 [True, True, True, True, True]
913 [True, True, True, True, True, True, True, True]
533 [True, True, True, True, True, True]
2199 [True, True, True]
921 [True, True, True, True]
794 [True, True, True]
283 [True, True, True, True]
925 [True, True, True, True, True]
1055 [True, True, True, True, True, True]
800 [True, True, True, True]
1057 [True, True, True, True, True]
674 [True, True, True, True, True, True, True, True]
162 [True, True, True]
33 [True, True, True, True, True]
164 [True, True, True, True, True]
675 [True, True, True, True]
551 [True, True, True, True, True]
1577 [True, True, True, True, True]
42 [True, True, True, True]
937 [True, True, True, True, True]
428 [True, True]
557 [True, True, True, True]
2222 [True, True, True, True]
308 [True, True, True, True]
949 [True, True, True]
950 [Tr